# Oblig 2a – Lineær og logistisk regresjon

**Våren 2026**

Det er en god idé å lese gjennom hele oppgavesettet før dere setter i gang. Dersom dere har spørsmål så kan dere:

- gå i gruppetime,
- spørre på Discourse
- eller sende epost til in1160-hjelp@ifi.uio.no dersom alternativene over av en eller annen grunn ikke passer for spørsmålet.

## Innlevering

Oppgaven leveres innen 11. mars klokken 23.59 i [Devilry](https://devilry.ifi.uio.no/).

Innleveringen skal bestå av én Jupyter notebook med både kode og tilhørende forklaringer. Før innlevering skal du kjøre gjennom hele notebooken, før du lagrer siste gang. Den bør kjøre uten å feile og vise alt som skal være med.

Vi understreker at innlevering av kode alene ikke er nok for å bestå oppgaven – vi forventer at notebooken også skal inneholde kommentarer (på norsk eller engelsk) på hva dere har gjort og begrunnelser for valgene dere har tatt underveis. La enhver oblig bli en trening i å formidle forskning. Bruk helst hele setninger, og matematiske formler om nødvendig. Resultater skal presenteres i tabeller på en oversiktlig måte. Det å forklare med egne ord, bruke begreper vi har gått gjennom på forelesningene og å forklare og reflektere over løsningene deres er en viktig del av læringsprosessen – ta det på alvor!

Når det gjelder bruk av generative prateroboter (ChatGPT og lignende): Dere kan bruke dem som en "sparringspartner", for eksempel for å forklare noe dere ikke helt har forstått. Dere har imidlertid ikke lov til å bruke dem til å generere løsninger (enten delvis eller fullstendig) til noen av oppgavene. Funksjoner for automatisk skriving av kode, som Copilot i VS Code, må derfor også være deaktivert mens dere jobber på obligen.

Bruker dere KI-verktøy vil vi også at dere kort beskriver hvordan dere har brukt dem under arbeidet med oppgaven.

Det er ikke mulighet for omlevering av obliger som ikke bestås.

**Poeng:** Obligen gir maksimalt 25 poeng og 6 bonuspoeng. For å bestå kreves det totalt 30 av 50 poeng fra oblig 2a og 2b tilsammen. Bonuspoengene telles ikke med i denne beregningen. Poengfordelingen er markert i overskriften til oppgaven, der `p` indikerer vanlig poeng og `b` indikerer bonuspoeng.


## Bakgrunn

I denne obligen skal vi jobbe med lineær og logistisk regresjon. Disse modellene er noen av de nyttigste maskinlæringsmodellene som finnes, der lineær regresjon blir brukt for *regresjon* og logistisk regresjon blir brukt til *klassifikasjon*. Oppgavene går ut på å først bli kjent med datasetene som blir brukt, og deretter implemetere litt funksjonalitet som tapsfunksjoner og prediksjon med modellene. For å slippe å implementere treningen av modellene skal vi bl.a. bruke modeller fra biblioteket `sklearn` til å trene på datasetene. Til slutt skal resultatet tolkes og det gjøres også litt trekk-seleksjon (feature selection).

Obligen er delt opp i mange mindre deloppgaver og det følger med tester for alle implementasjonsoppgavene. Testene printer ut feilmeldinger dersom testen feiler, og printer ikke ut noe om testen bestås (med mindre dere sender med argumentet `message_on_pass=True`). Feilmeldingene sjekker for flere typer feil, som feil returverdi, feil utregning, error under kjøring og liknende, så det kan være nyttig å lese feilmeldingen for å hjelpe til med implementeringen av oppgavene. Testene må bestås for at oppgavene skal bli godkjent, men beståtte tester garanterer ikke at implementasjonen er korrekt.

Oppgavene skal implementeres med NumPy og klassene `LinearRegression` og `LogisticRegression` fra `sklearn`. Det er ikke tillatt å importere funksjoner som gjør det oppgaven ber dere om å implementere, for eksempel å bruke `accuracy_score` fra `sklearn` for å implementere `calculate_accuracy`.

Vi starter med å importere de nødvendige bibliotekene:

In [1]:
import numpy as np
from sklearn.linear_model import LinearRegression, LogisticRegression

from tests2a import (
    test_calculate_accuracy,
    test_calculate_bce,
    test_calculate_mse,
    test_predict_linear_regression,
    test_predict_logistic_regression,
    test_sigmoid,
)
from utils2a import get_auto_mpg_data, get_spambase_data

## 1 - Lineær regresjon [Totalt 12p og 3b]

### 1.0 - Bakgrunn og dataset [2p, 1b]

Lineær regresjon er en enkel, men effektiv, regresjonmodell. Modellen predikerer et reelt tall (fra minus uendelig til pluss uendelig) gitt en liste med inputtrekk (features). Ved å bruke et datasett som inkluderer både input og tilhørende fasitverdier kan vi finne koeffisientene som minimerer forskjellen mellom prediksjonene og fasitverdiene.

Vi starter med å se på datasettet vårt. Vi skal bruke [`Auto MPG` datasettet](https://people.math.carleton.ca/~smills/2013-14/DataMining/Data/UCI%20Machine%20Learning%20Repository%20%20Auto%20MPG%20Data%20Set.htm), som er et mye brukt og konseptuelt relativt enkelt datasett for regresjon. Det inneholder informasjon om biler fra 70- og 80-tallet, som blir brukt til å predikere hvor bensineffektive bilene er, målt i *miles per gallon* (mpg). Vi skal først jobbe med to trekk: `weight`, som er vekten til bilene, og `acceleration`, som er hvor rask bilene kan akselerere. Litt senere i obligen vil vi inkludere flere trekk fra datasettet.

La oss først avklare notasjonen litt: Vi bruker $n$ til å notere antall datapunkter i datasettet, som er 398 for datasettet vårt, eller 392 etter at vi har fjernet datapunkter med manglende verdier. Hvert av datapunktene har input $\textbf{x}$ og sann verdi (target) $y$. Dette vil si vi har to lister av lengde $n$, input $\textbf{x}_1, \textbf{x}_2, ..., \textbf{x}_n$ og de sanne verdiene $y_1, y_2, ..., y_n$. Merk at input $\textbf{x}$ er notert med **fet skrift**, som representerer en vektor med flere tall. Vi starter med $p = 2$ antall trekk, som for bil nummer $i$ notert ved $\textbf{x}_i = (x_{i, 1}, x_{i, 2})$, der $x_{i, 1}$ representerer vekten til bil nummer $i$ og $x_{i, 2}$ representerer bil nummer $i$ sin akselerasjon. Vi bruker ofte indeksen $i$ til å representere et *vilkårlig* tall, som her kan betegne et tall mellom 1 og $n$.

Vi starter med å laste inn datasettet. Datasettet blir lest med biblioteket `seaborn`. Dere skal ikke endre på noen av argumentene i funksjonen under.

In [2]:
mpg_data = get_auto_mpg_data(
    columns_to_include=["weight", "acceleration"],
    val_ratio=0.2,
    test_ratio=0.2,
    perform_scaling=False,
)
x_train = mpg_data["x_train"]
y_train = mpg_data["y_train"]

**Merk**: Dataen over er ikke skalert (fra keyword argumentetet `perform_scaling=False`). Dette er for å bli bedre kjent med dataen i menneskelige lesbare måleenheter. Når vi bruker dataen til å trene og predikere senere er det viktig at dataen blir skalert, ved å gi inn argumentet `perform_scaling=True`.

Objektet som blir returnert, `mpg_data`, er en Python ordbok (dictionary) med trenings-, validerings- og testdatasplitter, samt navn på trekkene. Som vanlig kommer vi til å trene modellen med treningsdataene, bruke evalueringsdataene til å finne ut hvilke treningskonfigurasjoner som fungerer best, og spare testdataene helt til slutt for å gjøre et estimat på hvor bra modellen vår vil fungere. Til å begynne med skal vi kun se på treningsdataene.

Input-dataene, gitt ved `mpg_data["x_train"]`, er et to-dimensjonalt NumPy array. Radene representerer ulike datapunkter og kolonnene forskjellige trekk. Dere kan indeksere rad $i$ og kolonne $j$ med `x_train[i, j]`. NumPy-indeksering har mange nyttige funksjoner. I tillegg til å indeksere en enkel verdi, kan man *slice* en delmengde av arrayen. For å få kolonne nummer 1 for rad 3 til rad 8 kan man for eksempel bruke `x_train[3:9, 1]`, og for å få kolonne 0 for alle radene kan man bruke `x_train[:, 0]`. 

La oss gjøre noen oppgaver for å bli kjent med datasettet og NumPy-indeksering. Alle oppgavene forholder seg til treningsdatasplitten for Auto MPG og indeksering starter på 0. Alle svarene kan finnes ved NumPy-indeksering og funksjoner uten looping. Print ut svarene på et lesbart og pent format. Verdien til `mpg_data["features_names"]` har navn tilsvarende kolonnene i x-dataene (her er `weight` den nulte kolonnen og `acceleration` den første). Måleenhenten til vekt er "pounds" og måleenheten til akselerasjonen er antall sekunder bilen bruker på 0 til 60 "miles per hour".

**Oppgave 1.0**: 

- a) Hvor mye veier bil nummer 3?
- b) Hva er akselerasjonen til bil nummer 100?
- c) Hva er gjennomsnittlig vekt for bilene og gjennomsnittlig akselerasjon?
- d) Hva er det høyeste og laveste bensinforbruket (oppgitt i "miles per gallon")?
- e) (*Bonus*) Hva er gjennomsnittlig bensinforbruk for bilene som veier mer enn 3000 pounds?

(Skriv svarene dine her)

**Svar:** a) Bil 3 veier 2050.0 pounds

b) Bil 100 veier 21.0 miles per hour

c) Gjennomsnittlig vekt er 2965 pounds, og gjennomsnittlig akselerasjon er 15.61 miles per hour

d) Det høyeste bensinforbruket er 46.6 miles per gallon og det laveste er 9.0 miles per gallon

e) Gjennomsnittlig bensinforbruk av biler som veier mer enn 3000 pounds er 17.1 miles per gallon

In [4]:
#Koding brukt for å finne svar
print("a) Bil 3 veier " + str(x_train[2][0]) + " pounds")
print("b) Bil 100 veier " + str(x_train[99][1]) + " miles per hour")
print("c) Gjennomsnittlig vekt er " + str(round(np.mean(x_train[:, 0]))) + " pounds, og gjennomsnittlig akselerasjon er "
      + str(round(np.mean(x_train[:, 1]), 2)) + " miles per hour")
print("d) Det høyeste bensinforbruket er " + str(np.max(y_train)) + " miles per gallon og det laveste er " + str(np.min(y_train)) 
      + " miles per gallon")

"""
Metode for å finne all bilene som veier mer enn 3000 pounds:
lager en liste like lang som antall biler, der hver verdi er
3000. Sammenligner hvert element med vekten av hver bil, bruker
så det til å finne alle indeksene som gir True, som er bilene
som veier mer enn 3000 pounds. Bruker så det til å finne gjennomsnittlig
bensinbruk.
"""
weight_array = np.full(len(x_train), 3000)
comparison_array = np.greater(x_train[:, 0], weight_array)
index_list = np.argwhere(comparison_array)[:, 0]
avg_gasoline_use = np.mean(y_train[index_list])
print("e) Gjennomsnittlig bensinforbruk av biler som veier mer enn 3000 pounds er " + str(round(avg_gasoline_use, 1))
      + " miles per gallon")

a) Bil 3 veier 2050.0 pounds
b) Bil 100 veier 21.0 miles per hour
c) Gjennomsnittlig vekt er 2965 pounds, og gjennomsnittlig akselerasjon er 15.61 miles per hour
d) Det høyeste bensinforbruket er 46.6 miles per gallon og det laveste er 9.0 miles per gallon
e) Gjennomsnittlig bensinforbruk av biler som veier mer enn 3000 pounds er 17.1 miles per gallon


### 1.1 - Gjennomsnittlig kvadratfeil (mean squared error) [2p]

Modellen finner verdien på koeffisienten ved å minimere feilen på treningsdataene. Feilen måles mellom prediksjonene $\hat{\textbf{y}}$ (uttalt *y-hatt*) og fasitverdiene $\textbf{y}$.

Det er generelt mange måter å måle feil og forskjeller på, men for lineær regresjon bruker man som regel *gjennomsnittlig kvadratfeil*, som regel kalt *mean squared error* (MSE). Dette måler altså den gjennomsnittlige kvadratiske forskjellen mellom prediksjonene og fasitverdiene, altså:

$$\text{MSE}(\textbf{y}, \hat{\textbf{y}}) = \frac{1}{n} \sum_{i = 0}^n (y_i - \hat{y}_i)^2$$

**Notasjon**: Symbolet $\Sigma$ er den store greske bokstaven *sigma*, som representerer en sum. Her blir $\sum_{i = 0}^n (y_i - \hat{y}_i)^2 = (y_1 - \hat{y}_1)^2 + (y_2 - \hat{y}_2)^2 + ... + (y_n - \hat{y}_n)^2$. 

**Vektorisering:** Hele obligen kan løses enten med eller uten vektorisering. Vektorisering handler om å la lavnivå-biblioteker som NumPy håndtere iterering (løkker), istedenfor å loope eksplisitt i Python. Vektorisering gir mye raskere kode, siden itereringen skjer i C istedenfor Python, i tillegg til at bibliotekene kan håndtere passende datastrukturer, minneallokering og parallelisering. Koden blir også mer kortfattet, siden man kan erstatte løkker med funksjonskall. Dette kan både gjøre koden mer lesbar, men også føre til at det er vanskeligere å forstå hva koden egentlig gjør. Dere kan selv velge om dere vil bruke NumPy funksjoner (som `np.square()`, `np.mean()`, vektorprodukt og matrisemultiplikasjon) eller om dere vil iterere eksplisitt. Det er også mulig å vektorisere deler av koden, eller først skrive koden med løkker og deretter erstatte den med vektorisert kode. Dette er en god måte å lære å skrive effektiv kode og bedre forstå hva de vektoriserte funksjonene gjør.

**Oppgave 1.1**: Implementer funksjonen `calculate_mse()`, som tar to arrayer av samme lengde og regner ut og returnerer MSE-en.

In [5]:
def calculate_mse(y_data, predictions):
    """
    Calculates and returns the mean squared error (MSE).
    Except two arrays of the same length as input.

    Args:
        y_data (np.array): The true values as an array of numbers.
        predictions (np.array): The predictions as an array (of same length as `y_data`) of numbers.

    Returns:
        float: Value corresponding to the MSE.
    """
    difference = y_data - predictions
    diff_squared = difference**2
    return np.mean(diff_squared)


test_calculate_mse(input_function=calculate_mse)

**Tester**: Obligen inneholder testfunksjoner lagret i `test2a.py` som dere kan bruke til å teste implementasjonene deres. Beståtte tester vil ikke garantere at det ikke er noen som er feil, men dersom noen av testene feiler er det sikkert at noe ikke er riktig. Testene er importert og kalt i prekoden. De tar inn funksjonen de skal teste som argument og itererer over flere test-tilfeller. Dersom testene bestås vil de av konvensjon ikke gi noen output, men dersom dere har lyst på en bekreftelse at testene bestod kan dere sende med argumentet `message_on_pass=True`. Dere står også fritt til å prøve funksjonene selv og lage egne tester, men pass på at den endelige besvarelsen ikke inneholder for mye kode som ikke er direkte svar på oppgaven.

### 1.2 - Prediksjon [2p]

Vi kan nå implementere prediksjon for lineær regresjon. La oss starte med å visualisere lineær regresjon med figuren under.

<!-- markdownlint-disable-next-line MD033 -->
<img src="bilder/linear_regression_general_formula.png" alt="Linear regression diagram" width="600"/>

Modellen får $\textbf{x}$ som input og gir en prediksjon av den sanne verdien som output, notert ved $\hat{y}$. Med datasettet vårt får den altså bilenes vekt og akselerasjon som input og bruker dette til å predikere hvor mye bensin den bruker. Dette gjør den ved å gange inputet med *vekter* og plusse på et *konstantledd* (også kalt bias). For lineære modeller er vektene og konstantleddet ofte kalt *koeffisienter*, og vi bruker den grenske bokstaven $\beta$ (uttalt *beta*) til å representere dem. For et datapunkt $i$, vil modellen vår altså regne ut prediksjonen $\hat{y}_i = \beta_1 x_{i, 1} + \beta_2 x_{i, 2} + \beta_0$. Her er altså $\beta_1$ og $\beta_2$ vektene (eller koeffisientene) til henholdsvis bilenes vekt og akselerasjon, mens $\beta_0$ er konstantleddet (biaset). Koeffisientene er altså tall som blir ganget med inputet, som også er tall, og plusset sammen.


La oss implementere prediksjonen for lineær regresjon i en funksjon.

**Oppgave 1.2**: Implementer `linear_regression_predict()`. Funksjonen tar to argumenter, inputdataene og koeffisientene, og returnerer prediksjonene. Inputdataene er her en to-dimensjonal array av størrelse $[n, p]$, altså $n$ rader og $p$ kolonner, der hver rad tilsvarer et datapunkt og hver kolonne tilsvarer et trekk. Koeffisientene blir representert av en array med $p + 1$ verdier, der den nulte verdien er konstantleddet, mens elementet på indeks 1 er vekten for det første trekket, elementet på indeks 2 er vekten til det andre trekket, og så videre. Koeffisientene blir brukt på alle $n$ datapunktene og returnerer en array av lengde $n$ med tilsvarende prediksjoner.

Her er det forskjellige muligheter med hensyn på vektorisering. Vi må lage to (implisitte eller eksplisitte) løkker: en over datapunktene og en over trekkene. Det er mulig å vektorisere begge disse løkkene, man dere kan også iterere eksplisitt over begge, eller gjøre en kombinasjon. For best læringsutbytte kan det være lurt å første iterere eksplisitt og så prøve å erstatte det med vektorisert kode. Operatoren `@` kan brukes som matrisemultiplikasjon med NumPy-arrayer og `np.dot()` kan brukes for å få prikkproduktet, der `np.dot(a, b)` tilsvarer $\sum_{i=1}^n a_1 \cdot b_1$.

In [6]:
def linear_regression_predict(x_data, coefficients):
    """
    Given input data `x_data` of shape [n, p] (`n` datapoints and `p` features)
    and `coefficients` of shape [p + 1], returns the predictions the model
    gives as an [n]-length array.

    Args:
        x_data (np.array): [n, p]-shaped array of input data.
        coefficient (np.array): [p + 1]-shaped array of coefficient. Element number
            zero corresponds to the bias while element one to p corresponds to
            the weight for feature 1 to p.

    Returns:
        np.array: [n]-shaped data of corresponding predictions.
    """
    prediction = x_data@coefficients[1:] + np.full(len(x_data), coefficients[0])
    return prediction


test_predict_linear_regression(input_function=linear_regression_predict)

### 1.3 - Kjøring og evaluering av lineær regresjon [2p]

Vi skal nå bruke maskinlæringsbiblioteket `sklearn`, og mer spesifikt klassen `LinearRegression`, til å trene en regresjonsmodell på datasettet vårt. Modellen kan initialiseres med `model = LinearRegression()`. Deretter kan den trenes ved å kalle `model.fit(X=x_train, y=y_train)`, som vil finne koeffisientene som gjør prediksjonene for treningsdataen `x_train` så lik de sanne verdiene `y_train` som mulig. Deretter kan man bruke modellen til å predikere med `model.predict(X=x_val)`.

**Oppgave 1.3**: Tren modellen på treningsdataene og prediker $\hat{y}$-verdier for valideringsdataene. Bruk så `calculate_mse` til å regne ut gjennomsnittlig kvadratfeil mellom disse prediksjonene og de sanne verdiene i `y_val`. Rapporter resultatet på et leselig format.

In [7]:
mpg_data = get_auto_mpg_data(
    columns_to_include=["weight", "acceleration"],
    val_ratio=0.2,
    test_ratio=0.2,
    perform_scaling=True,
)
x_train = mpg_data["x_train"]
y_train = mpg_data["y_train"]
x_val = mpg_data["x_val"]
y_val = mpg_data["y_val"]

model = LinearRegression()
# TODO: Train (fit), predict on the validation set and calculate and report MSE.
model.fit(X=x_train, y=y_train)
predictions = model.predict(X=x_val)
print("Gjennomsnittlig kvadratfeil: " + str(round(calculate_mse(y_val, predictions), 2)))

Gjennomsnittlig kvadratfeil: 18.27


**Optimalisering av koeffisientene:** Når `.fit()` blir kalt blir koeffisientene satt til verdiene som minimerer MSE-en på treningsdatasplitten. Hvordan blir dette gjort? For lineær regresjon er det mulig å løse minimering av MSE som et likningsett av flere ukjente, der de ukjente er koeffisientene. Dette kan løses med grunnleggende lineær algebra, der den generelle løsningen kan uttrykkes som et enkelt regnestykke som bruker matrisemultiplikasjon og invertering. Her skiller lineær regresjon seg fra nesten alle andre maskinlæringsmodeller. Dersom modellen vår blir noe mer komplisert eller inkluderte veldig mange trekk, vil det ikke være mulig å uttrykke koeffisientene som minimerer tapet som et enkelt utrykk, og det brukes som regel iterative numeriske prosesser som *gradientnedstigning* (gradient descent) som det står et bonusavsnitt om i del 2.4 av denne obligen.

### 1.4 - Trekk-seleksjon (feature selection) [4p, 2b]

Så langt har vi brukt bilenes vekt og akselerasjon til å predikere bensinforbruket, men det originale datasettet inneholder flere trekk som vi kan bruke. Det siste vi skal gjøre med lineær regresjon er å prøve forskjellige trekk. Dette kalles *trekk-seleksjon* (feature selection).

Vi skal se på de fire følgende trekkene:
- `horsepower`: Antall hestekrefter bilene har.
- `model_year`: Året bilene ble produsert.
- `cylinders`: Antall sylindere i motoren til bilene.
- `displacement`: Volumet til motoren.

**Oppgave 1.4.1**: Tren (på treningsdataene) og evaluer modellen (på valideringsdataene) med trekkene `horsepower` og `model_year` i tillegg til `weight` og `acceleration`. Hvordan forandrer dette resultatet? Prøv å forklare ved å resonnere over hva trekkene representerer og hvordan dette kan relatere til bensinforbruk.

In [ ]:
mpg_data = get_auto_mpg_data(
    columns_to_include=["weight", "acceleration", "horsepower", "model_year"],
    perform_scaling=True,
)
x_train = mpg_data["x_train"]
y_train = mpg_data["y_train"]
x_val = mpg_data["x_val"]
y_val = mpg_data["y_val"]

model = LinearRegression()
model.fit(X=x_train, y=y_train)
predictions = model.predict(X=x_val)
print("Gjennomsnittlig kvadratfeil: " + str(round(calculate_mse(y_val, predictions), 2)))

Gjennomsnittlig kvadratfeil: 10.6


**Svar:** Gjennomsnittlig kvadratfeil blir nå bare 10.6 i motsetning til 18.27, så vi har fått et bedre resultat. Dette gir mening, da både hestekraft og når bilen ble laget kan påvirke bensinforbruket. Hestekraft er effekten av motoren, mens nyere modeller har større sansynlighet for å bruke bedre teknologi som kan gir lavere bensinforbruk.

**Oppgave 1.4.2**: Bruk nå `cylinders` og `displacement` i tillegg til de fire andre trekkene til å trene og evaluere modellen. Hvordan blir resultatet nå i forhold til de to forrige tilfellene? Vurder ulike forklaringer på hvorfor resultatet ble slik.

In [ ]:
mpg_data = get_auto_mpg_data(
    columns_to_include=[
        "cylinders",
        "displacement",
        "horsepower",
        "weight",
        "acceleration",
        "model_year",
    ],
    perform_scaling=True,
)
x_train = mpg_data["x_train"]
y_train = mpg_data["y_train"]
x_val = mpg_data["x_val"]
y_val = mpg_data["y_val"]

model = LinearRegression()
model.fit(X=x_train, y=y_train)
predictions = model.predict(X=x_val)
print("Gjennomsnittlig kvadratfeil: " + str(round(calculate_mse(y_val, predictions), 2)))

Gjennomsnittlig kvadratfeil: 11.61


**Svar:** Gjennomsnittlig kvadratfeil blir nå 11.61, litt værre enn vårt forrige resultat. Grunner til dette kan ha å gjøre med at modellen vår er nå overtilpasset siden vi trener den på for mange trekk, og at de nye trekkene ikke har like mye å si når det gjelder bensinforbruket.

**Oppgave 1.4.3** (*Bonus*): Prøv forskjellige kombinasjoner av trekk i datasettet og rapporter den laveste MSE-en du finner på valideringsdataene. Du kan bruke hvilken som helst tilnærming for å prøve ut trekk.

In [ ]:
columns = ["cylinders", "displacement", "horsepower", "weight", "acceleration", "model_year"]

"""Vi bruker combinations fra itertools for å finne alle mulige kombinasjoner av kolonner vi kan, så
trener vi modellen på alle mulige kombinasjoner og legger til MSE-en for hver kombinasjon i en ordbok."""
from itertools import combinations
comb = combinations(columns, 1)

min_mse = 100
best_columns = ["cylinders"]

for i in range(1, len(columns) + 1):
    comb = list(combinations(columns, i))
    for included_columns in comb:
        mpg_data = get_auto_mpg_data(columns_to_include=list(included_columns), perform_scaling=True,)
        x_train = mpg_data["x_train"]
        y_train = mpg_data["y_train"]
        x_val = mpg_data["x_val"]
        y_val = mpg_data["y_val"]

        model = LinearRegression()
        model.fit(X=x_train, y=y_train)
        predictions = model.predict(X=x_val)
        mse = calculate_mse(y_val, predictions)
        if min_mse > mse:
            min_mse = mse
            best_columns = list(included_columns)

print("Minste MSE: " + str(round(min_mse, 3)))
print("Beste mulige kolonner: " + str(best_columns))

Minste MSE: 9.265
Beste mulige kolonner: ['cylinders', 'displacement', 'acceleration', 'model_year']


**Svar:** Beste mulig MSE git 9.265, og de beste mulige trekkene er `cylinders`, `displacement`, `acceleration` og `model_year`. Det tidligere svaret om at `cylinders` og `displacement` ikke påvirker resultatet mye er altså feil, og mest sannsynlig fikk vi værre resultat på grunn av overtilpasning.

**Oppgave 1.4.4:** (*Bonus*) Bruk den beste modellen (modellen trent på trekkene som gir lavest valideringstap) til å lage prediksjoner for testdataene og rapporter tapet. Dere kan hente testdataene med `mpg_data["x_test"]` og `mpg_data["y_test"]`.

Grunnen til at vi gjør dette er å få et mer realistisk anslag på hvordan den endelige modellen vil prestere på usette datapunkter. Når vi trener mange modeller og velger den med lavest valideringsfeil, blir denne feilen et dårlig estimat for den faktiske generaliseringsfeilen, nettopp fordi modellen er valgt på grunn av sin lave valideringsfeil. Derfor holder vi av et eget testdatasett som ikke brukes i verken prosessering eller trening, slik at dataene er helt nye både for modellen og for dem som utvikler den. På den måten unngår vi modeller som presterer godt på trenings- og valideringsdata, men dårligere ellers - noe som ofte kan skje med kraftigere modeller som dype nevrale nettverk.

In [33]:
mpg_data = get_auto_mpg_data(columns_to_include=best_columns, perform_scaling=True,)
x_train = mpg_data["x_train"]
y_train = mpg_data["y_train"]
x_test = mpg_data["x_test"]
y_test = mpg_data["y_test"]

model = LinearRegression()
model.fit(X=x_train, y=y_train)
predictions = model.predict(X=x_test)
mse = calculate_mse(y_test, predictions)
print("MSE for prediksjoner på testdatasettet: " + str(round(mse, 3)))

MSE for prediksjoner på testdatasettet: 14.872


## Del 2 - Logistisk regresjon [Totalt 13p og 3b]

### 2.0 - Bakgrunn og datasett [2p, 1b]

I denne delen tar vi for oss binær klassifikasjon med *logistisk regresjon*. Mens lineær regresjon brukes for regresjonsoppgaver der målet er å forutsi reelle tall, brukes logistisk regresjon for klassifikasjonsoppgaver der målet er å tilordne eksempler til _ulike klasser_. Binær klassifikasjon handler om å klassifisere mellom to klasser, for eksempel "ja" eller "nei", eller "0" eller "1".

I binær klassifikasjon er de sanne verdiene $y_i$ enten lik 0 eller lik 1 (dette kan representere binære merkelapper som for eksempel "ikke-hund" og "hund"). Prediksjonene $\hat{y}_i$ er reelle tall mellom 0 og 1.

Selv om logistisk regresjon er en annen modell enn lineær regresjon, er det mange likheter. Det er derfor mulig å gjenbruke noe av koden og metodologien brukt i den første delen.

Datasettet vi skal bruke heter [*Spambase*](https://archive.ics.uci.edu/dataset/94/spambasefor), som er et datasett med trekk (features) fra 4601 e-poster som er markert som spam eller ikke-spam (de sanne verdiene). E-postene har totalt 57 trekk, der vi skal starte med en mindre delmengde av dem. Trekkene beskriver frekvensen av spesifikke ord, tegn og store bokstaver. Vi starter med følgende trekk:

- `word_freq_free`: Frekvensen av ordet `free` i e-posten.
- `char_freq_%24`: Frekvensen av tegnet `$` i e-posten (enkodingen `%24` representerer tegnet `$`).
- `capital_run_length_total`: Summen av antall store bokstaver i e-posten.

I datasettet er "frekvensen" av et ord målt ved antall `100` * `forekomster av ordet` / `totalt antall ord i e-posten` og tilsvarende for frekvensen av tegn.

Datasettet kan lastes med følgende:

In [34]:
spam_data = get_spambase_data(
    columns_to_include=["word_freq_free", "char_freq_%24", "capital_run_length_total"],
    val_ratio=0.2,
    test_ratio=0.2,
    perform_scaling=False,
)
x_train = spam_data["x_train"]
y_train = spam_data["y_train"]
x_val = spam_data["x_val"]
y_val = spam_data["y_val"]


**Oppgave 2.0**: Vi begynner med å undersøke datasettet. I alle deloppgavene under skal vi kun jobbe med dataene i treningssplitten. Navnet på trekkene kan hentes med `spam_data["feature_names"]`, og y-dataene angir spam-e-poster som `1` og ikke-spam som `0`.

- a) Hva er det høyeste antallet store bokstaver i en e-post?
- b) Hvor er antallet spam og ikke-spam e-poster?
- c) Hva er det høyeste frekvensen av tegnet `$` i en e-post?
- d) Hva er det høyeste frekvensen av tegnet `$` i en e-post som *ikke* er spam?
- e) [Bonus] Hva er gjennomsnittlig antall store bokstaver i spam e-poster, og gjennomsnittlig antall store bokstaver i ikke-spam e-poster?

In [35]:
print("a) Høyeste antallet store bokstaver: " + str(np.max(x_train[:, 2])))
print("b) Antallet spam: " + str(np.count_nonzero(y_train)) + ". Antallet ikke-spam: " + str(len(y_train) - np.count_nonzero(y_train)))
print("c) Høyste frekvens av tegnet $: " + str(np.max(x_train[:, 1])))

#Finner indeksene til alle e-postene som ikke er spam:
non_spam_indices = np.argwhere(np.logical_not(y_train))[:, 0]
print("d) Høyste frekvens av tegnet $ i en ikke-spam e-post: " + str(np.max(x_train[non_spam_indices, 1])))

spam_indices = np.argwhere(y_train)[:, 0]
print("e) Gjennomsnittlig antall store bokstaver i spam e-poster: " + str(round(np.mean(x_train[spam_indices][:, 2]))) +
      ". Gjennomsnittlig antall store bokstaver i ikke-spam e-poster: " + str(round(np.mean(x_train[non_spam_indices][:, 2]))))

a) Høyeste antallet store bokstaver: 15841.0
b) Antallet spam: 1094. Antallet ikke-spam: 1666
c) Høyste frekvens av tegnet $: 6.003
d) Høyste frekvens av tegnet $ i en ikke-spam e-post: 2.038
e) Gjennomsnittlig antall store bokstaver i spam e-poster: 481. Gjennomsnittlig antall store bokstaver i ikke-spam e-poster: 159


### 2.1 - Binær kryssentropi [3p]

I regresjon brukte vi tapsfunksjonen MSE, mens vi kommer til å bruke *binær kryssentropi* for logistisk regresjon, som oftest kalt *binary cross entropy* (BCE). Navnet "kryssentropi" kommer fra informasjonsteori, en annen gren innenfor informatikk, men funksjonen passer godt som tapsfunksjon for klassifikasjon.

La oss først se på hvordan og hvorfor BCE fungerer. Vi starter med definisjonen av BCE for et enkelt datapunkt $i$, som er:
$$
\begin{align*}
l_{\text{BCE}}(y_i, \hat{y}_i) & = 
\begin{cases}
- \log(1 - \hat{y}_i), & \text{hvis } y_i = 0, \\
- \log(\hat{y}_i), & \text{hvis } y_i = 1.
\end{cases} \\[8mm]
\text{\textrm{noe som kan også skrives:}} & \\[4mm]
&=  - \left[ y_i \cdot \log(\hat{y}_i) + (1 - y_i) \cdot \log(1 - \hat{y}_i)\right]
\end{align*}
$$

Det gjennomsnittlige tapet for et helt datasett og en BCE-tapsfunksjon er dermed:

$$\begin{align*} \hat{L}_{\text{BCE}}(\textbf{y}, \hat{\textbf{y}}) & = \frac{1}{n} \sum_{i=1}^{n} l_{\text{BCE}}(y_i, \hat{y}_i) \\
& = - \frac{1}{n} \sum_{i=1}^{n} \left[ y_i \cdot \log(\hat{y}_i) + (1 - y_i) \cdot \log(1 - \hat{y}_i)\right] \end{align*}$$

La oss kort se på `log()`-delen, altså logaritmen. Generelt sett er logaritmer definert som den inverse funksjonen av potenser, men vi trenger kun å se på tilfellet der inputet er mellom 0 og 1. Da er det to ting å legge merke til: $\log(1) = 0$, og $\log(x)$ nærmerer seg minus uendelig når $x$ nærmer seg 0. Dette minuset er opphavet til minuset i starten av formelen for BCE.

**Oppgave 2.1.1**: Her er noen veiledende spørsmål for å bli kjent med BCE.

- a) For et gitt datapunkt, anta at $y = 0$. Hva blir BCE dersom modellen predikerer nøyaktig 0?
- b) Hva omtrent blir BCE dersom modellen predikerer et tall veldig nærme 1?
- c) Anta så at $y = 1$. Hva blir BCE dersom modellen predikerer nøyaktig 1?
- d) Hva blir BCE om modellen predikerer et tall veldig nærme 0?
- e) Prøv å beskrive i én setning hvorfor de fire egenskapene over er gunstig for en tapsfunksjon.

**Hint**: Ukesoppgavene i uke 6 går igjennom BCE i nærmere detalj.

a) Hvis modellen predikerer nøyaktig 0, så vil $l_\text{BCE}$ gi $0$.

b) $l_\text{BCE}$ gir en veldig høy verdi for prediksjoner veldig nærme $1$, og gir høyere verdier jo nærmere prediksjonen er $1$.

c) Hvis modellen predikerer nøyaktig $1$, så vil $l_\text{BCE}$ gi $0$.

d) Lignende som for b), så vil prediksjoner nær $0$ gi høye tall, med høyere verdier jo nærmere prediksjonen er $0$.

**Oppgave 2.1.2**: Implementer funksjonen `calculate_bce()`, som gitt to arrayer av samme lengde regner ut binær kryssentropi (BCE) av de to arrayene. Bruk `np.log()` for logaritmen. Som med MSE kan dere velge om dere vil vektorisere koden eller ikke.

In [ ]:
def calculate_bce(y_data, predictions):
    """
    Returns the binary cross entropy (BCE) of the input values.

    Args:
        y_data (np.array): [n]-shaped array of true values (zeros or ones).
        predictions (np.array): [n]-shaped array of predictions (between zero and one).

    Returns:
        float: The BCE.
    """
    l_BCE = y_data*np.log(predictions) + (1 - y_data)*np.log(1 - predictions)
    
    """Vi får noen problemer med nan-verdier senere når prediksjonene gir 1, så vi håndterer det her.
    Fikser det ved å anta at den sanne verdien i disse tilfellene er 1 og at prediksjoner gir
    egentlig en verdi veldig lik 1, som gjør at vi kan anta at (1-y_data)*np.log(1-predictions)
    delen gir 0."""
    indices = np.where(np.isnan(l_BCE) == True)

    for i in indices[0]:
        l_BCE[i] = 0

    return -np.mean(l_BCE)


test_calculate_bce(input_function=calculate_bce, message_on_pass=True)


Passed: `test_calculate_bce`. All [5/5] tests passed. 


### 2.2 - Prediksjon [2p]

Prediksjon er noe forskjellig for lineær regresjon og logistisk regresjon. For lineær regresjon ganget vi vektene med de tilsvarende trekkene og plusset på konstantleddet, som kan gi et tall mellom minus og pluss uendelig. For logistisk regresjon gjør vi først det samme steget, men så putter vi dette tallet inn i en *sigmoid* funksjon som gir et tall mellom 0 og 1.

Sigmoid-funksjonen, som ofte er representert ved $\sigma$ (det greske symbolet uttalt "sigma"), er definert ved:

$$\sigma(x) = \frac{1}{1 + e^{-x}}$$

Her er $e$ den matematiske konstanten [*Eulers konstant*](https://en.wikipedia.org/wiki/E_(mathematical_constant)), men i praksis spiller det liten rolle hvilket tall som blir brukt, så det er ikke nødvendig å ha noe særlig forståelse for dette tallet. En ting som kan hjelpe med forståelsen er at $e^{-x}$ går mot $0$ når $x$ er veldig høyt og går mot uendelig når $x$ har en høy negativ verdi. Det medfører at *$\sigma(x)$ blir nærme 0 når x har en høy negativ verdi og nærme 1 når $x$ har en høy positiv verdi*.

**Oppgave 2.2.1**: Implementer funksjonen `sigmoid()` som regner ut sigmoid-funksjonen. Bruk `np.exp()` for potensen, der `np.exp(x)` $ = e^x$.

In [37]:
def sigmoid(values):
    """
    Calculates the sigmoid function of the input, sigmoid(x) = 1 / (1 + e^(-x))

    Args:
        values (np.array): [n]-shaped array of values to send to the sigmoid function.

    Returns:
        np.array: [n]-shaped array of float values corresponding to the output of the sigmoid function.
    """
    return 1/(1 + np.exp(-values))


test_sigmoid(input_function=sigmoid)

Prediksjon for logistisk regresjon blir altså å først regne ut vektene ganger trekkene pluss konstantleddet, slik som med lineær regresjon, og så putte dette inn i sigmoid-funksjonen. Vektene ganger trekkene pluss konstantleddet blir ofte notert med $z$, kalt *den vektede summen* (eng: *weighted sum*), og vi kan skrive formelen for prediksjon som:

$$\hat{y} = \sigma(z) = \sigma\left(\sum_{i=1}^n \beta_i \cdot x_i + \beta_0\right)$$

Vi kan visualisere logistisk regresjon i følgende diagram:

<!-- markdownlint-disable-next-line MD033 -->
<img src="bilder/logistic_regression_general_formula.png" alt="Linear regression diagram" width="600"/>

**Oppgave 2.2.2** Implementer funksjonen `logistic_regression_predict()`. Som med lineær regresjon skal metoden ta inn en to-dimensjonal array av dimensjon $[n, p]$, der hver rad er et datapunkt og kolonnene tilsvarer trekk i inputen, og returnere en vektor av lengde $[n]$ med prediksjoner basert på formelen over. Bruk `sigmoid` implementasjonen deres fra oppgave 2.2.1.  
**Tips**: Det er mulig å gjenbruke mye av koden fra lineær regresjon, det er kun nødvendig å legge til sigmoid-funksjonen.

In [38]:
def logistic_regression_predict(x_data, coefficients):
    """
    Given input data `x_data` of shape [n, p] (`n` datapoints and `p` features)
    and `coefficients` of shape [p + 1], returns the predictions the model
    gives as an [n]-length array.

    Args:
        x_data (np.array): [n, p]-shaped array of input data.
        coefficient (np.array): [p + 1]-shaped array of coefficient. Element number
            zero corresponds to the bias while element one to p corresponds to
            the weight for feature 1 to p.

    Returns:
        np.array: [n]-shaped data of corresponding predictions.
    """
    prediction = sigmoid(x_data@coefficients[1:] + np.full(len(x_data), coefficients[0]))
    return prediction


test_predict_logistic_regression(input_function=logistic_regression_predict)

### 2.3 - Kjøring og evaluering av logistisk regresjon [3p]

Det er på tide å kjøre modellen vår på datasettet og klassifisere e-poster som spam eller ikke-spam.

**Oppgave 2.3.1**: Tren modellen på treningsdatasettet, prediker på evalueringsdataene og rapporter BCE-en. Dere kan bruke `model = LogisticRegression()` og `model.fit()` som med lineær regresjon. For å få sigmoid-verdiene (prediksjonene før de er rundet av til 0 eller 1) kan dere bruke `model.predict_proba(X=x_val)[:, 1]`, siden `model.predict()` vil returnere klassifikasjonene, altså verdiene etter de er rundet av til 0 eller 1.

In [104]:
spam_data = get_spambase_data(
    columns_to_include=["word_freq_free", "char_freq_%24", "capital_run_length_total"],
    val_ratio=0.2,
    test_ratio=0.2,
    perform_scaling=True,
)
x_train = spam_data["x_train"]
y_train = spam_data["y_train"]
x_val = spam_data["x_val"]
y_val = spam_data["y_val"]

model = LogisticRegression()
# TODO: Train (fit) model on training set, predict probabilities on the evaluation set and report the BCE.
X = model.fit(X=x_train, y=y_train)
predictions = model.predict_proba(X=x_val)[:, 1]
print("The BCE: " +str(round(calculate_bce(y_val, predictions), 3)))

The BCE: 0.46


C:\Users\af64\AppData\Local\Temp\ipykernel_2656\3738163942.py:12: RuntimeWarning: divide by zero encountered in log
  a = y_data*np.log(predictions) + (1 - y_data)*np.log(1 - predictions)
C:\Users\af64\AppData\Local\Temp\ipykernel_2656\3738163942.py:12: RuntimeWarning: invalid value encountered in multiply
  a = y_data*np.log(predictions) + (1 - y_data)*np.log(1 - predictions)


Det neste steget er å evaluere modellen med *nøyaktighet* (accuracy). Nøyaktighet regner ut forholdet mellom korrekt klassifiserte datapunkter og totalt antall datapunkter. Med sanne verdier $\textbf{y} = (y_1, y_2, ..., y_n)$ og klassifikasjoner $\textbf{k} = (k_1, k_2, ..., k_n)$ (der hver $y_i$ og $k_i$ er enten 0 eller 1), kan vi beregne nøyaktighet som:

$$\text{accuracy}(\textbf{y}, \textbf{k}) = \frac{1}{n} \sum_{i=1}^n I(y_i = k_i)$$

der $I$ er identitetsfunksjonen, som returnerer 1 dersom argumentet er sant, og 0 ellers.

Prediksjonene fra `model.predict_proba()` returnerer verdiene fra sigmoid-funksjonen, men for å regne ut nøyaktighet må verdiene være nøyaktig lik 0 eller 1. Det kan man enkelt få ved å runde av verdier til 0 eller 1 med en terskelverdi, for eksempel 0.5. Dette gjør metoden `model.predict()`.

**Oppgave 2.3.2** Implementer funksjonen `calculate_accuracy()` som tar inn to binære arrayer av samme lengde og returnerer nøyaktigheten. Rapporter deretter nøyaktigheten til modellen fra oppgave 2.2.1 på valideringsdataene. Hadde du vært fornøyd med en algoritme for å klassifisere spam med denne nøyaktigheten?

In [102]:
def calculate_accuracy(y_data, classifications):
    """
    Calculates accuracy on provided input.

    Args:
        y_data (np.array): [n]-shaped array of true values.
        classifications (_type_): [n]-shaped array of classifications.

    Returns:a
        int: The accuracy for the inputs.
    """
    return np.mean(np.equal(y_data, classifications)*1)


test_calculate_accuracy(input_function=calculate_accuracy)

### 2.4 - Fullt datasett og tolkning [3p, 2b]

Hittil har vi bare brukt 3 av de 57 trekkene i datasettet. Med den generelle implementasjonen av klassen vår kan vi derimot bruke et vilkårlig antall trekk. La oss prøve å trene modellen med alle trekkene.

**Oppgave 2.4.1**: Tren den logistiske modellen med alle trekkene i datasettet og rapporter nøyaktigheten. Hadde dette vært tilstrekkelig for å bruke modellen til å klassifisere spam i praksis?

In [ ]:
spam_data = get_spambase_data(
    columns_to_include=None,  # Corresponds to all features
    val_ratio=0.2,
    test_ratio=0.2,
    perform_scaling=True,
)
x_train = spam_data["x_train"]
y_train = spam_data["y_train"]
x_val = spam_data["x_val"]
y_val = spam_data["y_val"]

model = LogisticRegression()
model.fit(X=x_train, y=y_train)
predictions = model.predict(X=x_val)
accuracy = calculate_accuracy(y_val, predictions)
print("Nøyaktighet av å trene med alle trekk: " + str(round(accuracy, 3)))

Nøyaktighet av å trene med alle trekk: 0.927


**Svar:** Vi får høy nøyaktighet, men det er tvilsomt at dette hadde vært nyttig for å klassifisere spam. Å trene modellen på så mange som 57 forskjellige trekk kan lett føre til overtilpasning, som ikke hadde ført til en modell som kunne godt klassifisere spam.

**Oppgave 2.4.2**: La oss til slutt reflektere litt over resultatet til modellen.

a) Anta at modellen for spam-klassifisering ble implementert i et ekte e-postfilter. Det er to måter å filtrere e-poster feil på: den ene er å klassifisere spam som ikke-spam, den andre er å klassifisere ikke-spam som spam. Synes du de to typene feil er like dårlige for filteret i praksis, eller vil en type feil være mer problematisk enn den andre? Forklar kort hvorfor du synes det.

b) Hvordan kunne du ha endret modellen slik at man kan prioritere å unngå den ene typen feil over den andre?

c) (*Bonus*) Datasettet vårt består av 57 trekk som allerede er manuelt bearbeidet gjennom såkalt _feature engineering_. Har du forslag til andre trekk som kunne vært inkludert for å bedre predikere om e-poster er spam eller ikke?

**Svar:** a) Det er værre å klassifisere ikke-spam som spam, da det kan føre til at muligens viktige e-poster blir klassifisert som spam. Det at spam blir klassifisert som ikke-spam derimot har ikke like store konsekvenser, da det kun fører til at brukeren må slette en uønsket e-post.

b) En mulighet kunne ha vært å endre grensen for når modellen er sikker på at noe er spam eller ikke-spam. F.eks hvis en verdi >= 0.5 vil vanligvis klassifiseres som spam, så kan vi istedet sette det til >= 0.75, som gjør at modellen må være mer sikker før den bestemmer om en e-post er spam eller ikke.

**Oppgave 2.4.3** (*Bonus*): Tren modellen på forskjellige kombinasjoner av trekk og finn kombinasjonen som gir lavest tap på valideringsdataene. Rapporter nøyaktigheten på testdataene til den kombinasjonen som ga lavest valideringstap.

In [105]:
from openml.datasets import get_dataset
test_data = get_dataset(44, version=1)
x_data, y_data, _, all_feature_names = test_data.get_data(target="class")

## Oppsummering

I denne obligen har vi sett på lineær og logistisk regresjon, der vi har implementert funksjonalitet som tapsfunksjoner og prediksjon, trent modeller med `sklearn` og tolket resultatet. Selv om både lineær og logistisk regresjon er enkle modeller, er de fortsatt svært effektive og mye brukte. Med god forbehandling av data og trekk kan lineær og logistisk regresjon gi mer enn tilstrekkelig ytelse, og til og med overgå mer komplekse modeller, spesielt med små datasett. De kan også bli brukt for å evaluere andre metoder ved å bruke resultatet som en *baseline* som kan sammenliknes med andre modellers resultat.

I faget IN2160 implementeres lineær og logistisk regresjon helt fra bunnen av, inkludert optimalisering av koeffisientene med gradientnedstigning.
